# Streamflow preprocessing

This notebook applies the streamflow preprocessing workflow implemented in `hydroevents`.

The preprocessing step cleans the original hourly streamflow series and prepares the input data for baseflow separation and event-based rainfall–runoff analysis.

## Method overview

The preprocessing workflow includes:

1. conversion of streamflow values to numeric format;
2. conversion of negative values to missing values;
3. conversion of short zero-flow sequences to missing values;
4. classification of missing-data gaps according to their duration;
5. filling of short gaps using a local centered mean, computed from the valid streamflow values surrounding each gap;
6. filling of intermediate gaps using shape-preserving interpolation;
7. preservation of long gaps as missing values;
8. aggregation of the processed streamflow series to daily mean values.

The resulting outputs are:

- a processed hourly streamflow series;
- a daily streamflow series;
- a summary of the cleaning and gap-filling operations.

## Import

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

from hydroevents import preprocess_streamflow

## Load input data

The input dataset must contain:

- `Date`: datetime column
- `Q`: streamflow values

In [ ]:
file_path = Path(r"path\to\input_file.xlsx")

df = pd.read_excel(file_path)
df.head()

## Define preprocessing parameters

The preprocessing workflow uses the following parameters:

- `ZERO_THRESHOLD`: maximum duration of zero-flow sequences treated as missing values;
- `MAX_GAP`: maximum gap length filled using a local centered mean;
- `MAX_GAP_INTERP`: maximum gap length filled using interpolation;
- `WINDOW_SIZE`: number of neighbouring time steps used to compute the local centered mean.
- `INTERPOLATION_METHOD`: interpolation method used for intermediate gaps (e.g. "linear", "pchip", or "spline").

All durations are expressed in number of time steps (e.g. hours for hourly streamflow data).

In [ ]:
DATE_COL = "Date"
Q_COL = "Q"

ZERO_THRESHOLD = 24
MAX_GAP = 48
MAX_GAP_INTERP = 120
WINDOW_SIZE = 5
INTERPOLATION_METHOD = 'pchip'

## Apply streamflow preprocessing

In [ ]:
df_processed, df_daily, summary = preprocess_streamflow(
    df,
    date_col=DATE_COL,
    q_col=Q_COL,
    apply_gap_filling=True,
    zero_threshold=ZERO_THRESHOLD,
    max_gap=MAX_GAP,
    max_gap_interp=MAX_GAP_INTERP,
    window_size=WINDOW_SIZE,
    interp_method = INTERPOLATION_METHOD
)

## Interactive visualization

The original and processed streamflow series are compared using an interactive plot.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_processed[DATE_COL],
        y=df_processed[f"{Q_COL}_original"],
        mode="lines",
        name="Original Q",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_processed[DATE_COL],
        y=df_processed[Q_COL],
        mode="lines",
        name="Processed Q",
    )
)

fig.update_layout(
    title="Original vs processed streamflow",
    xaxis_title="Date",
    yaxis_title="Q",
    template="plotly_white",
)

fig.show()

## Daily streamflow series

The processed hourly streamflow series is also aggregated to daily mean values.  
This daily series can be used for Master Recession Curve estimation.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_daily[DATE_COL],
        y=df_daily[Q_COL],
        mode="lines",
        name="Daily Q",
    )
)

fig.update_layout(
    title="Daily streamflow series",
    xaxis_title="Date",
    yaxis_title="Daily mean Q",
    template="plotly_white",
)

fig.show()

## Save output

In [ ]:
output_dir = Path(r"path\to\output")
# output_dir = Path("../examples/data")

output_dir.mkdir(parents=True, exist_ok=True)

base_name = file_path.stem

processed_file = output_dir / f"{base_name}_processed.xlsx"
daily_file = output_dir / f"{base_name}_daily.xlsx"

df_processed.to_excel(processed_file, index=False)
df_daily.to_excel(daily_file, index=False)

print(f"Processed streamflow saved to: {processed_file.resolve()}")
print(f"Daily streamflow saved to: {daily_file.resolve()}")